In [0]:

bronze_table = "job_market.bronze.hacker_news_api_raw"

spark.table(bronze_table).printSchema()


In [0]:

from pyspark.sql import functions as F

df = spark.readStream.table(bronze_table)

df = (
    df.select(
        F.col("item.id").cast("long").alias("post_id"),
        F.col("thread_id").cast("long").alias("thread_id"),
        F.col("thread_month"),
        F.col("item.parent").cast("long").alias("parent_id"),
        F.col("item.type").alias("item_type"),
        F.col("item.by").alias("author"),
        F.col("item.text").alias("text_html"),
        F.timestamp_seconds(F.col("item.time")).alias("posted_at"),
        F.coalesce(F.col("item.deleted"), F.lit(False)).alias("is_deleted"),
        F.coalesce(F.col("item.dead"), F.lit(False)).alias("is_dead"),
        F.col("fetched_at").cast("timestamp").alias("fetched_at"),
        F.col("source_file"),
        F.col("ingested_at"),
        F.col("_rescued_data")
    )
    .withColumn("posted_year", F.year("posted_at"))
    .withColumn("posted_month", F.month("posted_at"))
    .withColumn(
        "text_plain",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.coalesce(F.col("text_html"), F.lit("")),
                    r"<[^>]*>",
                    " "
                ),
                r"\s+",
                " "
            )
        )
    )
)


In [0]:

df = df.withColumn(
    "quarantine_reason",
    F.concat_ws(
        " | ",
        F.when(
            F.col("post_id").isNull() | (F.col("post_id") <= 0),
            "INVALID_POST_ID"
        ),
        F.when(
            F.col("parent_id").isNull() |
            (F.col("parent_id") != F.col("thread_id")),
            "INVALID_PARENT_ID"
        ),
        F.when(F.col("item_type") != "comment", "INVALID_ITEM_TYPE"),
        F.when(F.col("is_deleted"), "DELETED_COMMENT"),
        F.when(F.col("is_dead"), "DEAD_COMMENT"),
        F.when(F.col("text_plain") == "", "MISSING_TEXT"),
        F.when(F.col("posted_at").isNull(), "MISSING_POST_TIME"),
        F.when(F.col("_rescued_data").isNotNull(), "SCHEMA_MISMATCH")
    )
)

df = df.withColumn(
    "is_quarantined",
    F.col("quarantine_reason") != ""
)


In [0]:

valid = (
    df.filter(~F.col("is_quarantined"))
    .dropDuplicates(["post_id"])
)

invalid = df.filter(F.col("is_quarantined"))


In [0]:

checkpoint_root = (
    "/Volumes/job_market/landing/raw_files/"
    "_pipeline_state/"
)

valid_table = "job_market.silver.hacker_news_api_posts"
quarantine_table = "job_market.silver.hacker_news_api_quarantine"


In [0]:

q1 = (
    valid.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_root + "hn_api_silver_valid_v1"
    )
    .trigger(availableNow=True)
    .toTable(valid_table)
)

q1.awaitTermination()


In [0]:

q2 = (
    invalid.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_root + "hn_api_silver_quarantine_v1"
    )
    .trigger(availableNow=True)
    .toTable(quarantine_table)
)

q2.awaitTermination()


In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM job_market.silver.hacker_news_api_posts)
     AS valid_posts,

    (SELECT COUNT(*)
     FROM job_market.silver.hacker_news_api_quarantine)
     AS quarantined_posts;


In [0]:
%sql

SELECT
    post_id,
    quarantine_reason,
    is_deleted,
    is_dead,
    text_html
FROM job_market.silver.hacker_news_api_quarantine
LIMIT 20;
